In [1]:
import os
os.environ["OPENAI_API_KEY"] = ""


In [2]:
import os
import json
import openai
import time
import asyncio
import csv
from dotenv import load_dotenv
from phi.agent import Agent
from fastmcp import Client

load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")


class DeliberateMongoAgent(Agent):
    """
    Deliberate Agent Pattern

    PLAN:
      - Understand intent
      - Decide tool
      - Produce validated arguments

    EXECUTE:
      - Call MCP tool

    EVALUATE:
      - Validate result
      - Retry with improved plan if needed
    """

    def __init__(
        self,
        client: Client,
        name="Deliberate MCP Agent",
        max_plan_retries: int = 2
    ):
        super().__init__(name=name)
        self.client = client
        self.max_plan_retries = max_plan_retries

    # --------------------------------------------------
    # PLAN PHASE
    # --------------------------------------------------
    async def plan(self, user_input: str, feedback: str | None = None):
        prompt = f"""
You are a planning agent for MongoDB queries.

Task:
Convert the user request into a VALID JSON filter
for `mongo_read_tool`.

Rules:
- Output ONLY JSON
- No explanation text
- Use YYYY-MM-DD for dates
- If nothing applies, return empty filter

Collection schema:
{{
  "_id": "ObjectId",
  "type": "string",
  "serial": "string",
  "make": "string",
  "warranty_start": "date",
  "warranty_end": "date",
  "purchase_showroom": "string",
  "city": "string",
  "cost": "float",
  "shipping_cost": "float",
  "purchase_date": "date"
}}

Previous feedback (if any):
{feedback}

User query:
"{user_input}"

Output format:
{{ "tool": "mongo_read_tool", "args": {{ "filter": {{ ... }} }} }}
"""

        response = openai.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        content = response.choices[0].message.content.strip()
        plan = json.loads(content)

        return plan["tool"], plan["args"]

    # --------------------------------------------------
    # EXECUTE PHASE
    # --------------------------------------------------
    async def execute(self, tool_name: str, args: dict):
        return await self.client.call_tool(tool_name, args)

    # --------------------------------------------------
    # EVALUATE PHASE
    # --------------------------------------------------
    def evaluate(self, tool_result):
        structured = getattr(tool_result, "structured_content", {})
        data = structured.get("result", {}).get("result")

        if data is None:
            raise ValueError("Empty result from tool")

        return data

    # --------------------------------------------------
    # RUN DELIBERATE LOOP
    # --------------------------------------------------
    async def run_nlp(self, user_input: str):
        start_time = time.time()
        feedback = None

        for attempt in range(self.max_plan_retries + 1):
            try:
                # ---------- PLAN ----------
                tool_name, args = await self.plan(
                    user_input=user_input,
                    feedback=feedback
                )

                # ---------- EXECUTE ----------
                result_stream = await self.execute(tool_name, args)

                # ---------- EVALUATE ----------
                result = self.evaluate(result_stream)

                elapsed = round(time.time() - start_time, 4)
                return {
                    "result": result,
                    "time_seconds": elapsed,
                    "plan_retries": attempt
                }

            except Exception as e:
                feedback = str(e)

                if attempt >= self.max_plan_retries:
                    elapsed = round(time.time() - start_time, 4)
                    return {
                        "result": None,
                        "error": feedback,
                        "time_seconds": elapsed,
                        "plan_retries": attempt
                    }

                await asyncio.sleep(0.3)


# -----------------------------
# Jupyter Notebook Usage
# -----------------------------
async def notebook_demo(user_input):
    server_url = "http://localhost:8000/mcp"

    async with Client(server_url) as mcp_client:
        await mcp_client.ping()
        print(f"Connected to MCP server at {server_url}")

        agent = DeliberateMongoAgent(client=mcp_client)

        result = await agent.run_nlp(user_input)
        print(f"Input: {user_input}")
        print(f"Result: {result.get('result')}")
        print(f"Time: {result.get('time_seconds')} sec")


async def run_all_queries():
    server_url = "http://localhost:8000/mcp"
    input_file = "db_nl_queries.txt"
    output_file = "results_Deliberate1.csv"

    with open(input_file, "r", encoding="utf-8") as f:
        queries = [q.strip() for q in f if q.strip()]

    with open(output_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["query", "time_seconds", "error", "plan_retries"])

    async with Client(server_url) as mcp_client:
        await mcp_client.ping()
        agent = DeliberateMongoAgent(client=mcp_client)

        for q in queries:
            result = await agent.run_nlp(q)
            print(result)

            with open(output_file, "a", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerow([
                    q,
                    result.get("time_seconds"),
                    result.get("error", ""),
                    result.get("plan_retries")
                ])

            print(f"✔ {q} | ⏱ {result.get('time_seconds')} sec")


# In Jupyter
# asyncio.run(run_all_queries())
await run_all_queries()


{'result': [{'_id': '68f8f5018d60a0237e5e8bac', 'type': 'TV', 'serial': 'ABC-12345', 'make': 'LG', 'warranty_start': '2025-10-21T00:00:00', 'warranty_end': '2025-10-31T00:00:00', 'purchase_showroom': 'PAI', 'city': 'DELHI', 'cost': 15000, 'shipping_cost': 500, 'purchase_date': '2025-10-22T00:00:00', 'createdAt': '2025-10-22T15:15:13.608000', 'updatedAt': '2025-10-22T15:15:13.608000', '__v': 0}, {'_id': '68f8f7d38d60a0237e5e8bb3', 'type': 'TV', 'serial': 'S-118294', 'make': 'LG', 'warranty_start': '2024-06-02T00:00:00', 'warranty_end': '2026-06-02T00:00:00', 'purchase_showroom': 'Croma', 'city': 'Delhi', 'cost': 68322.74, 'shipping_cost': 2682.64, 'purchase_date': '2024-06-02T00:00:00', 'createdAt': '2025-10-22T15:27:15.388000', 'updatedAt': '2025-10-22T15:27:15.388000', '__v': 0}, {'_id': '68f8f7d38d60a0237e5e8bc2', 'type': 'TV', 'serial': 'S-533408', 'make': 'LG', 'warranty_start': '2025-04-19T00:00:00', 'warranty_end': '2027-04-19T00:00:00', 'purchase_showroom': 'Vijay Sales', 'city'